# Hierarchical Document Index (hindex) Visualizer

Explore the hierarchy trees built by **hindex** — Colette's vision-first hierarchical
indexing strategy.  Reads directly from the on-disk JSON trees and the HDF5 image
kvstore.  No service initialization required.

Covers:
- Document & node statistics
- Interactive sunburst / treemap hierarchy view
- Page coverage (Gantt-style chart per document)
- Node depth and size distributions
- Node summary text statistics
- Page images from the kvstore for a selected tree node
- Cross-document comparison (when multiple documents are indexed)

## 1. Setup

In [1]:
# Install optional packages if needed
# !pip install plotly pandas pillow h5py

In [2]:
import json
import uuid
from pathlib import Path
from io import BytesIO

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
from PIL import Image
import h5py

print('Packages loaded.')


Packages loaded.


## 2. Configuration

In [3]:
import os

# Path to your Colette app directory (contains mm_index/)
colette_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
app_dir = Path(os.path.join(colette_root, 'app_colette'))

hindex_dir = app_dir / 'mm_index' / 'hindex'
kvstore_path = app_dir / 'kvstore.db'

print(f'hindex dir : {hindex_dir}')
print(f'  exists   : {hindex_dir.exists()}')
print(f'kvstore    : {kvstore_path}')
print(f'  exists   : {kvstore_path.exists()}')

hindex dir : /data1/ldejesus/repos/public/colette/app_colette/mm_index/hindex
  exists   : True
kvstore    : /data1/ldejesus/repos/public/colette/app_colette/kvstore.h5
  exists   : False


## 3. Load Hierarchy Trees

In [4]:
import sys
sys.path.insert(0, os.path.join(colette_root, 'src'))
from colette.backends.hf.hindex import DocumentHierarchy

json_files = sorted(hindex_dir.glob('*.json')) if hindex_dir.exists() else []
hierarchies = []
for p in json_files:
    try:
        data = json.loads(p.read_text(encoding='utf-8'))
        hierarchies.append(DocumentHierarchy.from_dict(data))
    except Exception as e:
        print(f'  WARNING: could not load {p.name}: {e}')

print(f'Loaded {len(hierarchies)} document hierarchy/ies')
for h in hierarchies:
    print(f'  - {h.doc_name}  ({h.page_count} pages, {len(h.flat_nodes())} nodes)')


Loaded 1 document hierarchy/ies
  - /data1/ldejesus/repos/public/colette/docs/pdf/COLETTEv2_Restitution_2025_03_07_v0.3_JB_light.pdf  (16 pages, 6 nodes)


## 4. Overview Statistics

In [5]:
def node_depth(node, depth=1):
    if not node.children:
        return depth
    return max(node_depth(c, depth + 1) for c in node.children)

rows = []
for h in hierarchies:
    flat = h.flat_nodes()
    max_depth = max((node_depth(n) for n in h.nodes), default=0)
    top_nodes = len(h.nodes)
    total_nodes = len(flat)
    with_summary = sum(1 for n in flat if n.summary)
    rows.append({
        'Document': h.doc_name,
        'Pages': h.page_count,
        'Top-level nodes': top_nodes,
        'Total nodes': total_nodes,
        'Max depth': max_depth,
        'Nodes with summary': with_summary,
        'Description': h.doc_description[:80] + '…' if len(h.doc_description) > 80 else h.doc_description,
    })

df_overview = pd.DataFrame(rows)
df_overview


,Document,Pages,Top-level nodes,Total nodes,Max depth,Nodes with summary,Description
0,/data1/ldejesus/repos/public/colette/docs/pdf/...,16,6,6,1,6,"This document is a presentation about COLETTE,..."


## 5. Interactive Hierarchy Tree (Sunburst)

Each ring represents one level of the document hierarchy.  Sector size = number of pages covered by that node.

In [6]:
def build_sunburst_data(hierarchy):
    ids, labels, parents, values, customdata = [], [], [], [], []
    root_id = hierarchy.doc_name
    ids.append(root_id)
    labels.append(f'📄 {Path(hierarchy.doc_name).name}')
    parents.append('')
    values.append(hierarchy.page_count)
    customdata.append(hierarchy.doc_description)

    def _walk(nodes, parent_id):
        for n in nodes:
            node_id = f'{root_id}/{n.node_id}'
            ids.append(node_id)
            labels.append(n.title)
            parents.append(parent_id)
            values.append(max(1, n.end_page - n.start_page + 1))
            customdata.append(
                f'Pages {n.start_page}–{n.end_page}\n{n.summary}' if n.summary
                else f'Pages {n.start_page}–{n.end_page}'
            )
            _walk(n.children, node_id)

    _walk(hierarchy.nodes, root_id)
    return ids, labels, parents, values, customdata


for h in hierarchies:
    ids, labels, parents, values, customdata = build_sunburst_data(h)
    fig = go.Figure(go.Sunburst(
        ids=ids, labels=labels, parents=parents, values=values,
        customdata=customdata,
        hovertemplate='<b>%{label}</b><br>%{customdata}<extra></extra>',
        branchvalues='total',
        maxdepth=4,
    ))
    fig.update_layout(
        title=f'Document Hierarchy — {Path(h.doc_name).name}',
        width=700, height=700,
    )
    fig.show()


### Treemap alternative

Same data as a treemap — easier to read for large trees.

In [7]:
for h in hierarchies:
    ids, labels, parents, values, customdata = build_sunburst_data(h)
    fig = go.Figure(go.Treemap(
        ids=ids, labels=labels, parents=parents, values=values,
        customdata=customdata,
        hovertemplate='<b>%{label}</b><br>%{customdata}<extra></extra>',
        branchvalues='total',
        maxdepth=4,
    ))
    fig.update_layout(
        title=f'Document Hierarchy (Treemap) — {Path(h.doc_name).name}',
        width=900, height=500,
    )
    fig.show()


## 6. Page Coverage

Gantt-style chart showing which pages belong to which section.

In [8]:
for h in hierarchies:
    flat = h.flat_nodes()
    # build rows: only leaf nodes (no children) or all nodes
    rows = []
    for n in flat:
        rows.append(dict(
            Node=f'[{n.node_id}] {n.title[:40]}',
            Start=n.start_page,
            Finish=n.end_page + 1,
            Pages=n.end_page - n.start_page + 1,
        ))
    df_gantt = pd.DataFrame(rows)

    fig = px.bar(
        df_gantt, x='Pages', y='Node', orientation='h',
        base='Start',
        title=f'Page Coverage per Node — {Path(h.doc_name).name}',
        labels={'Start': 'Start page', 'Pages': 'Number of pages'},
        hover_data=['Start', 'Finish', 'Pages'],
        color='Pages',
        color_continuous_scale='Blues',
    )
    fig.update_layout(height=max(300, len(rows) * 28 + 100), yaxis={'categoryorder': 'trace'})
    fig.show()


## 7. Node Statistics

In [9]:
all_rows = []
for h in hierarchies:
    def _depth(node, d=1):
        return d if not node.children else max(_depth(c, d+1) for c in node.children)

    for n in h.flat_nodes():
        all_rows.append(dict(
            document=Path(h.doc_name).name,
            node_id=n.node_id,
            title=n.title,
            pages=n.end_page - n.start_page + 1,
            depth=_depth(n),
            has_children=bool(n.children),
            summary_words=len(n.summary.split()) if n.summary else 0,
        ))

df_nodes = pd.DataFrame(all_rows)

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Pages per Node', 'Summary Length (words)'])

for doc_name, grp in df_nodes.groupby('document'):
    fig.add_trace(go.Histogram(x=grp['pages'], name=doc_name, opacity=0.7,
                               nbinsx=20), row=1, col=1)
    fig.add_trace(go.Histogram(x=grp['summary_words'], name=doc_name, opacity=0.7,
                               nbinsx=20, showlegend=False), row=1, col=2)

fig.update_layout(title='Node-level Statistics', barmode='overlay', height=400)
fig.show()

print(df_nodes[['document', 'pages', 'depth', 'summary_words']].describe().round(2))


       pages  depth  summary_words
count   6.00    6.0           6.00
mean    2.67    1.0          37.33
std     3.14    0.0           5.85
min     1.00    1.0          32.00
25%     1.00    1.0          32.75
50%     1.50    1.0          35.50
75%     2.00    1.0          41.25
max     9.00    1.0          46.00


## 8. Page Images for a Selected Node

Fetch and display the actual page images for any node using the HDF5 kvstore.

In [10]:
# ── Configuration ───────────────────────────────────────────
TARGET_DOC_INDEX = 0   # index into `hierarchies`
TARGET_NODE_ID   = None  # e.g. '0002'; None → first top-level node
MAX_IMAGES       = 4   # cap so the notebook doesn't scroll forever

# ── Resolve target ───────────────────────────────────────────
h = hierarchies[TARGET_DOC_INDEX]
flat = h.flat_nodes()

if TARGET_NODE_ID is None:
    node = h.nodes[0]
else:
    node = next((n for n in flat if n.node_id == TARGET_NODE_ID), h.nodes[0])

print(f'Document : {h.doc_name}')
print(f'Node     : [{node.node_id}] {node.title}')
print(f'Pages    : {node.start_page} → {node.end_page}')
if node.summary:
    print(f'Summary  : {node.summary}')
print(f'kvstore  : {kvstore_path}')


Document : /data1/ldejesus/repos/public/colette/docs/pdf/COLETTEv2_Restitution_2025_03_07_v0.3_JB_light.pdf
Node     : [0001] Introduction et Présentation
Pages    : 0 → 1
Summary  : This document introduces Colette, a technical training course for GenAI-oriented applications. Held at CNES on March 7, 2025, it is co-organized by Airbus and CNES, focusing on practical AI implementation in aerospace and defense sectors.
kvstore  : /data1/ldejesus/repos/public/colette/app_colette/kvstore.h5


In [11]:
# ── Load images from kvstore ────────────────────────────────
from colette.kvstore import HDF5ImageStorage
from IPython.display import display as ipy_display

if not kvstore_path.exists():
    print(f'kvstore not found at {kvstore_path}. Run service_index first.')
else:
    kvstore = HDF5ImageStorage(kvstore_path, mode='r')
    page_range = range(node.start_page, min(node.end_page + 1, node.start_page + MAX_IMAGES))
    loaded = []
    missing = []
    for page_idx in page_range:
        if page_idx >= len(h.page_keys):
            missing.append(page_idx)
            continue
        key = h.page_keys[page_idx]
        # Full-page images may be absent for slides the layout detector saw as single-region.
        # Fall back to the first crop key in that case.
        resolved_key = key if kvstore.has_key(key) else f'{key}_crop_0000'
        try:
            img = kvstore.retrieve_image(resolved_key)
            loaded.append((page_idx, img, resolved_key))
        except KeyError:
            missing.append(page_idx)
    kvstore.close()

    if missing:
        print(f'Pages not in kvstore (will be skipped): {missing}')

    n_imgs = len(loaded)
    if n_imgs:
        cols = min(2, n_imgs)
        rows_count = (n_imgs + cols - 1) // cols
        fig, axes = plt.subplots(rows_count, cols, figsize=(cols * 7, rows_count * 9))
        axes_flat = [axes] if n_imgs == 1 else list(axes.flat)
        for ax, (page_idx, img, rkey) in zip(axes_flat, loaded):
            ax.imshow(img)
            suffix = '' if rkey == h.page_keys[page_idx] else ' (crop)'
            ax.set_title(f'Page {page_idx}{suffix}  [{node.node_id}] {node.title[:45]}',
                         fontsize=9)
            ax.axis('off')
        for ax in axes_flat[n_imgs:]:
            ax.set_visible(False)
        plt.suptitle(f'{Path(h.doc_name).name} — {node.title}', fontsize=11, y=1.01)
        plt.tight_layout()
        plt.show()
    else:
        print('No images found in kvstore for this node.')

kvstore not found at /data1/ldejesus/repos/public/colette/app_colette/kvstore.h5. Run service_index first.


## 9. Cross-document Comparison

Radar chart comparing structural properties across indexed documents.

In [12]:
if len(hierarchies) < 2:
    print('Index more than one document to see cross-document comparison.')
else:
    def _doc_stats(h):
        flat = h.flat_nodes()
        def _depth(n, d=1): return d if not n.children else max(_depth(c, d+1) for c in n.children)
        depths = [_depth(n) for n in h.nodes]
        pages_per_node = [n.end_page - n.start_page + 1 for n in flat]
        return dict(
            name=Path(h.doc_name).name,
            total_nodes=len(flat),
            max_depth=max(depths) if depths else 0,
            avg_pages_per_node=round(float(np.mean(pages_per_node)), 1) if pages_per_node else 0,
            nodes_with_summary=sum(1 for n in flat if n.summary),
            page_count=h.page_count,
        )

    stats = [_doc_stats(h) for h in hierarchies]
    df_cmp = pd.DataFrame(stats).set_index('name')
    display(df_cmp)

    # normalize for radar
    df_norm = (df_cmp - df_cmp.min()) / (df_cmp.max() - df_cmp.min() + 1e-9)
    categories = list(df_norm.columns)
    fig = go.Figure()
    for doc in df_norm.index:
        vals = df_norm.loc[doc].tolist()
        fig.add_trace(go.Scatterpolar(
            r=vals + [vals[0]],
            theta=categories + [categories[0]],
            fill='toself', name=doc
        ))
    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
        title='Cross-document Structural Comparison (normalized)',
        height=500,
    )
    fig.show()


Index more than one document to see cross-document comparison.


## 10. Export Hierarchy Summary

In [13]:
# Export all node data to CSV
# df_nodes.to_csv('hindex_nodes.csv', index=False)
# print('Saved hindex_nodes.csv')

# Print compact text tree for any document
for h in hierarchies:
    print('=' * 60)
    print(h.as_text_tree())
    print()


Document: /data1/ldejesus/repos/public/colette/docs/pdf/COLETTEv2_Restitution_2025_03_07_v0.3_JB_light.pdf (16 pages)
[0001] Introduction et Présentation (pages 0–1)
[0002] RAG et papier original (pages 2–2)
[0003] RAG en application (pages 3–4)
[0004] Objectifs pour Colette (pages 5–5)
[0005] Vision-RAG de Colette (pages 6–6)
[0006] Pages 8–16 (pages 7–15)

